# Evaluación Práctica Machine Learning: Identificación de Razas Bovinas con CNN

**Asignatura:** Machine Learning - 8vo Semestre, INACAP  
**Objetivo:** Desarrollar, entrenar y justificar técnicamente un modelo de Deep Learning capaz de clasificar de manera automatizada las **3 razas bovinas predominantes en la zona centro-sur y sur de Chile** (**Angus**, **Hereford** y **Holstein**).

> **Delimitación de Dominio Operativo:** Se excluyeron del dataset razas foráneas o no representativas de los campos de la zona (como Brahman o Texas Longhorn). Esto simplifica la frontera de decisión, reduce falsos positivos y focaliza la capacidad de extracción de características de la red en los biotipos lecheros y carniceros reales de la región.

---

## 0. Prerrequisito Fundamental: Levantamiento del Entorno Virtual (`venv`)

### ¿Por qué levantar un entorno virtual ANTES de instalar dependencias o ejecutar el código?
En entornos profesionales y en ingeniería de software aplicada a Machine Learning, **nunca se deben instalar paquetes globales directamente en el sistema operativo**. Levantar un entorno virtual es mandatorio por las siguientes razones técnicas:

1. **Aislamiento de Dependencias y Evitación de Conflictos (*Dependency Hell*):**
   - Bibliotecas como `tensorflow`, `numpy` y `pillow` tienen dependencias estrictas de versiones C/C++ subyacentes. Un entorno virtual garantiza que las versiones no colisionen con otros proyectos o con los paquetes del sistema operativo.
2. **Protección del Sistema Operativo (PEP 668):**
   - En distribuciones Linux modernas y servidores de producción, el gestor de paquetes del sistema (`pacman`, `apt`) protege los módulos de Python del sistema contra sobreescrituras accidentales de `pip` (`externally-managed-environment`).
3. **Reproducibilidad Académica y Profesional:**
   - Permite que el profesor o cualquier evaluador pueda reproducir el experimento exactamente con las mismas versiones sin alterar su computadora.

### Instrucciones para levantar el entorno desde la terminal:
```bash
# 1. Situarse en la carpeta del proyecto
cd "evaluacion ML"

# 2. Crear el entorno virtual aislado (ejecutar una sola vez)
python3 -m venv venv

# 3. Activar el entorno virtual
source venv/bin/activate       # En Linux / macOS
# venv\Scripts\activate         # En Windows (PowerShell/CMD)

# 4. Actualizar pip e instalar dependencias dentro del entorno
pip install --upgrade pip
pip install tensorflow numpy matplotlib pillow ipykernel

# 5. (Opcional) Registrar el entorno en Jupyter para seleccionarlo como Kernel
python -m ipykernel install --user --name=venv_ml --display-name "Python (ML Venv)"
```

> **Nota:** Una vez activado el entorno, el prompt de la terminal mostrará el prefijo `(venv)`. A partir de ese momento, cualquier instalación de paquetes queda confinada de forma segura dentro de la carpeta `venv/`.

---
## 1. Justificación Técnica: ¿Por qué se hizo con ESTO y no con LO OTRO?

Para esta evaluación, la arquitectura y metodología elegida se sustenta en tres decisiones de diseño críticas:

### A. ¿Por qué Redes Neuronales Convolucionales (CNN) y NO un Perceptrón Multicapa Tradicional (Red Densa / MLP)?

| Aspecto Crítico | Red Densa Tradicional (MLP) | Red Neuronal Convolucional (CNN) - *Elegida* |
| :--- | :--- | :--- |
| **Tratamiento de la Imagen** | **Destruye la forma 2D** mediante `Flatten()` al inicio. Convierte la matriz de 128×128×3 en un vector plano de 49.152 números desconectados, perdiendo la relación de vecindad entre píxeles. | **Preserva la topología 2D**. Escanea la imagen con tensores y filtros locales de 3×3 píxeles que respetan la geometría del animal. |
| **Volumen de Parámetros** | **Explosión combinatoria:** Conectar 49.152 entradas a una capa de 128 neuronas generaría más de **6.2 millones de parámetros** en una sola capa. Con un dataset pequeño, el modelo simplemente memorizaría el ruido (*overfitting* severo). | **Compartición de Pesos (*Weight Sharing*):** Un filtro de 3×3 tiene solo 27 parámetros entrenables que se reutilizan deslizándose por toda la foto, reduciendo los parámetros drásticamente a unas pocas decenas de miles. |
| **Invarianza Espacial** | **Rígida y dependiente de coordenadas:** Si la cabeza blanca de un vacuno Hereford aparece en la esquina superior izquierda en vez del centro, el MLP no la reconoce porque sus conexiones están atadas a posiciones fijas de píxeles. | **Invariante a la Traslación:** El filtro que aprende a detectar la cabeza blanca o las manchas del Holstein se activa independientemente de la posición del animal dentro del encuadre. |
| **Extracción de Rasgos** | No posee jerarquía visual; intenta correlacionar píxeles crudos aislados con la clase. | **Jerarquía Espacial Profunda:** Las primeras capas detectan bordes y contrastes; las capas medias detectan texturas y manchas de pelaje; las capas profundas reconocen biotipos corporales (lechero vs. cárnico). |

### B. ¿Por qué Deep Learning de Extremo a Extremo (End-to-End) y NO un Enfoque Basado en Reglas o Algoritmos Clásicos?
- **Inviabilidad del Enfoque por Reglas:** En visión artificial biológica bajo condiciones de campo (potreros en Chile), es imposible programar reglas condicionales del tipo *«si el pixel X es negro y el pixel Y es blanco, entonces es Holstein»*. La variabilidad lumínica (sol directo, nublado en el sur), sombras, pasto, barro y diferentes posturas del animal quiebran cualquier heurística manual.
- **Aprendizaje de Representaciones:** La CNN aprende automáticamente cuáles son los patrones discriminantes (*feature representation*) directamente a partir de las imágenes de entrenamiento sin requerir ingeniería manual de características (*hand-crafted features*).

### C. ¿Por qué Clasificación Multiclase con `Categorical Crossentropy` + `Softmax` y Delimitación de Razas Locales?
- En el laboratorio inicial se utilizó una salida binaria (`sigmoid`, 1 neurona) adecuada solo para dos clases (perros vs. gatos).
- Para la realidad ganadera de la zona, donde coexisten razas de carne y leche (**Angus**, **Hereford** y **Holstein**), se implementa una capa final con 3 neuronas activadas por **`Softmax`**.
- `Softmax` proyecta las salidas en una **distribución probabilística normalizada** cuya suma es estrictamente $1.0$ (100%), permitiendo conocer no solo la raza ganadora, sino la certidumbre e incertidumbre del diagnóstico.
- **Manejo del conjunto de pruebas:** Se conservan en la carpeta `pruebas/` únicamente las fotos correspondientes a las razas entrenadas (`angus`, `hereford`, `holstein`) para una verificación justa y consistente de la matriz de evaluación.

In [ ]:
# =====================================================================
# PASO 1: IMPORTACIÓN DE BIBLIOTECAS Y PREPARACIÓN DEL ENTORNO
# =====================================================================
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.utils import load_img, img_to_array
import os

# Semilla pseudoaleatoria fija para garantizar la reproducibilidad de la evaluación
tf.random.set_seed(42)
np.random.seed(42)

print(f"Versión de TensorFlow en uso: {tf.__version__}")
print("Dispositivos físicos disponibles:", tf.config.list_physical_devices())

In [ ]:
# =====================================================================
# PASO 2: CARGA Y ESTRUCTURACIÓN DEL DATASET BOVINO
# =====================================================================
RUTA_DATASET = "dataset"
TAMANO_IMAGEN = (128, 128)  # Resolución óptima para balancear detalle morfológico y velocidad en CPU
BATCH_SIZE = 8              # Lotes pequeños para actualización frecuente de gradientes

# Carga automática infiriendo etiquetas desde las subcarpetas
dataset_entrenamiento = tf.keras.utils.image_dataset_from_directory(
    RUTA_DATASET,
    image_size=TAMANO_IMAGEN,
    batch_size=BATCH_SIZE,
    label_mode="categorical",  # Codificación One-Hot para clasificación multiclase
    shuffle=True,
    seed=42
)

clases_razas = dataset_entrenamiento.class_names
num_clases = len(clases_razas)

print("\n" + "="*50)
print(f"Dataset cargado exitosamente.")
print(f"Número total de razas a clasificar: {num_clases}")
print(f"Nombres de las clases: {clases_razas}")
print("="*50)

In [ ]:
# =====================================================================
# PASO 3: CONSTRUCCIÓN DE LA ARQUITECTURA DE LA CNN
# =====================================================================
modelo_bovino = models.Sequential([
    # Capa de Entrada con dimensiones definidas
    layers.Input(shape=(128, 128, 3)),
    
    # 1. Normalización de Intensidades de Píxel [0, 255] -> [0.0, 1.0]
    layers.Rescaling(1./255),
    
    # 2. Pipeline de Data Augmentation para robustez frente a condiciones de potrero
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    
    # --- BLOQUE CONVOLUCIONAL 1: Detección de Primitivas Visuales ---
    layers.Conv2D(32, (3, 3), activation="relu"),  # 32 filtros de 3x3
    layers.MaxPooling2D(pool_size=(2, 2)),        # Reduce ancho y alto a la mitad (64x64)
    
    # --- BLOQUE CONVOLUCIONAL 2: Texturas de Pelaje y Patrones de Color ---
    # Aprende a diferenciar manchas Holstein, cara blanca de Hereford y uniformidad de Angus
    layers.Conv2D(64, (3, 3), activation="relu"),  # 64 filtros de 3x3
    layers.MaxPooling2D(pool_size=(2, 2)),        # Reduce a (32x32)
    
    # --- BLOQUE CONVOLUCIONAL 3: Morfología Compleja y Rasgos Biológicos ---
    # Captura cuernos (Longhorn), giba y orejas caídas (Brahman) y biotipo lechero vs cárnico
    layers.Conv2D(128, (3, 3), activation="relu"), # 128 filtros de 3x3
    layers.MaxPooling2D(pool_size=(2, 2)),        # Reduce a (16x16)
    
    # --- CLASIFICADOR DENSO FINAL ---
    layers.Flatten(),                              # Aplana los mapas de características a 1D
    layers.Dense(128, activation="relu"),          # Capa densa de razonamiento no lineal
    layers.Dropout(0.3),                           # Regularización: apaga 30% de neuronas aleatoriamente
    layers.Dense(num_clases, activation="softmax") # Probabilidad normalizada por cada una de las 5 razas
], name="Clasificador_Razas_Bovinas_CNN")

# Compilación con optimizador Adam y pérdida de entropía cruzada categórica
modelo_bovino.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# Resumen detallado de la arquitectura
modelo_bovino.summary()

In [ ]:
# =====================================================================
# PASO 4: ENTRENAMIENTO DEL MODELO Y GUARDADO EN DISCO
# =====================================================================
EPOCAS = 25

print(f"Iniciando entrenamiento del modelo durante {EPOCAS} épocas...")
historial = modelo_bovino.fit(
    dataset_entrenamiento,
    epochs=EPOCAS
)

# Guardado del modelo en el formato nativo moderno de Keras
NOMBRE_MODELO = "modelo_bovinos.keras"
modelo_bovino.save(NOMBRE_MODELO)
print(f"\n[OK] Modelo entrenado y serializado exitosamente en: {NOMBRE_MODELO}")

In [ ]:
# =====================================================================
# PASO 5: VISUALIZACIÓN DE CURVAS DE RENDIMIENTO (ACCURACY & LOSS)
# =====================================================================
plt.figure(figsize=(14, 5))

# Gráfico de Precisión (Accuracy)
plt.subplot(1, 2, 1)
plt.plot(historial.history["accuracy"], label="Precisión de Entrenamiento", color="#1b5e20", linewidth=2, marker="o")
plt.title("Evolución de la Precisión (Accuracy) por Época", fontsize=13, fontweight="bold")
plt.xlabel("Época", fontsize=11)
plt.ylabel("Accuracy (0.0 a 1.0)", fontsize=11)
plt.ylim([0, 1.05])
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend(fontsize=11)

# Gráfico de Pérdida (Loss)
plt.subplot(1, 2, 2)
plt.plot(historial.history["loss"], label="Pérdida (Categorical Crossentropy)", color="#b71c1c", linewidth=2, marker="s")
plt.title("Evolución de la Función de Pérdida (Loss)", fontsize=13, fontweight="bold")
plt.xlabel("Época", fontsize=11)
plt.ylabel("Loss", fontsize=11)
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend(fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# =====================================================================
# PASO 6: INFERENCIA Y EVALUACIÓN CON IMÁGENES DE PRUEBA DE LA ZONA
# =====================================================================
# Cargar el modelo guardado para verificar persistencia e inferencia desacoplada
modelo_cargado = tf.keras.models.load_model(NOMBRE_MODELO)

def evaluar_imagen_bovina(ruta_imagen):
    """
    Carga una imagen nueva de prueba, la preprocesa y genera la distribución de probabilidad
    sobre las 3 razas bovinas entrenadas en el sistema (Angus, Hereford, Holstein).
    """
    # Cargar y redimensionar al tamaño esperado por la red
    img = load_img(ruta_imagen, target_size=TAMANO_IMAGEN)
    # Convertir a arreglo numpy y agregar la dimensión de batch (1, 128, 128, 3)
    x = np.expand_dims(img_to_array(img), axis=0)
    
    # Predecir probabilidades con Softmax
    probabilidades = modelo_cargado.predict(x, verbose=0)[0]
    idx_predicho = np.argmax(probabilidades)
    raza_ganadora = clases_razas[idx_predicho]
    certeza = probabilidades[idx_predicho] * 100
    
    # Mostrar resultado e imagen
    plt.figure(figsize=(4, 4))
    plt.imshow(img)
    plt.axis("off")
    plt.title(f"Diagnóstico: {raza_ganadora.upper()}\nConfianza: {certeza:.1f}%", fontsize=12, fontweight="bold", color="darkblue")
    plt.show()
    
    print(f"\n[+] Archivo evaluado: {os.path.basename(ruta_imagen)}")
    print(f"    -> Raza Identificada: {raza_ganadora.upper()} ({certeza:.2f}% de certeza)")
    print("    Desglose probabilístico comparativo:")
    for raza, prob in zip(clases_razas, probabilidades):
        barra = "#" * int(prob * 25)
        print(f"      * {raza.ljust(12)}: {prob*100:6.2f}% | {barra}")

# =====================================================================
# EVALUACIÓN DE LAS IMÁGENES DE PRUEBA INDEPENDIENTES (pruebas/)
# Las fotos evaluadas corresponden estrictamente a las 3 razas de la zona.
# =====================================================================
CARPETA_PRUEBAS = "pruebas"
if os.path.exists(CARPETA_PRUEBAS):
    fotos_prueba = sorted([f for f in os.listdir(CARPETA_PRUEBAS) if f.lower().endswith((".jpg", ".jpeg", ".png"))])
    print(f"Evaluando {len(fotos_prueba)} imágenes de prueba independientes (Angus, Hereford, Holstein):\n")
    for foto in fotos_prueba:
        ruta_completa = os.path.join(CARPETA_PRUEBAS, foto)
        print("-" * 55)
        evaluar_imagen_bovina(ruta_completa)
else:
    print(f"Aviso: La carpeta '{CARPETA_PRUEBAS}' no fue encontrada.")